In [0]:
storage_account = 'salespipelineadls'
storage_key = 'YOUR_KEY_HERE'

spark.conf.set(
    f'fs.azure.account.key.{storage_account}.dfs.core.windows.net',
    storage_key
)

silver_path   = f'abfss://silver@{storage_account}.dfs.core.windows.net/sales_orders/'
gold_revenue  = f'abfss://gold@{storage_account}.dfs.core.windows.net/monthly_revenue/'
gold_products = f'abfss://gold@{storage_account}.dfs.core.windows.net/top_products/'

print('Paths configured.')

Paths configured.


In [0]:
from pyspark.sql import functions as F

# Read silver Delta table
df_silver = spark.read.format('delta').load(silver_path)

print(f'Silver rows available: {df_silver.count()}')

# Show a quick summary
display(df_silver.groupBy('status').count())

Silver rows available: 10000


status,count
Shipped,1985
Processing,1367
Cancelled,4204
Delivered,2444


In [0]:
# Filter out cancelled orders — they don't generate revenue
df_active = df_silver.filter(F.col('status') != 'Cancelled')

# Group by year, month, region and calculate metrics
df_monthly = df_active \
    .groupBy('order_year', 'order_month', 'region') \
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.count('order_id').alias('order_count'),
        F.round(F.avg('revenue'), 2).alias('avg_order_value'),
        F.countDistinct('customer_id').alias('unique_customers'),
        F.sum('quantity').alias('total_units_sold')
    ) \
    .withColumn('report_month',
        F.concat(
            F.col('order_year').cast('string'),
            F.lit('-'),
                        F.lpad(F.col('order_month').cast('string'), 2, '0')
        )
    ) \
    .withColumn('load_ts', F.current_timestamp()) \
    .orderBy('order_year', 'order_month', 'region')

print(f'Monthly revenue rows: {df_monthly.count()}')
display(df_monthly.limit(20))

Monthly revenue rows: 96


order_year,order_month,region,total_revenue,order_count,avg_order_value,unique_customers,total_units_sold,report_month,load_ts
2022,1,East,5975106.00,90,66390.07,20,512,2022-01,2026-04-16T07:24:13.7Z
2022,1,North,2292276.50,51,44946.60,17,293,2022-01,2026-04-16T07:24:13.7Z
2022,1,South,4175829.00,47,88847.43,18,268,2022-01,2026-04-16T07:24:13.7Z
2022,1,West,3239373.00,30,107979.10,15,187,2022-01,2026-04-16T07:24:13.7Z
2022,2,East,5063232.50,97,52198.27,20,571,2022-02,2026-04-16T07:24:13.7Z
2022,2,North,4319403.50,43,100451.24,19,249,2022-02,2026-04-16T07:24:13.7Z
2022,2,South,4886433.50,50,97728.67,17,284,2022-02,2026-04-16T07:24:13.7Z
2022,2,West,816040.00,24,34001.67,15,133,2022-02,2026-04-16T07:24:13.7Z
2022,3,East,6761159.00,96,70428.74,19,548,2022-03,2026-04-16T07:24:13.7Z
2022,3,North,3692764.00,42,87922.95,18,245,2022-03,2026-04-16T07:24:13.7Z


In [0]:
# Aggregate by product — which products sell most
df_products = df_active \
    .groupBy('product_id') \
    .agg(
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.sum('quantity').alias('total_units_sold'),
        F.count('order_id').alias('order_count'),
        F.round(F.avg('revenue'), 2).alias('avg_order_value'),
        F.countDistinct('customer_id').alias('unique_buyers')
    ) \
    .orderBy(F.desc('total_revenue')) \
    .withColumn('rank', F.row_number().over(
        __import__('pyspark.sql.window', fromlist=['Window'])
        .Window.orderBy(F.desc('total_revenue'))
    )) \
    .withColumn('load_ts', F.current_timestamp())

print(f'Products: {df_products.count()}')
display(df_products)

Products: 10


product_id,total_revenue,total_units_sold,order_count,avg_order_value,unique_buyers,rank,load_ts
1,218964750.00,3223,562,389616.99,20,1,2026-04-16T07:25:14.671Z
4,65435040.00,3290,600,109058.40,20,2,2026-04-16T07:25:14.671Z
7,55411200.00,3398,605,91588.76,20,3,2026-04-16T07:25:14.671Z
3,22850125.00,2975,536,42630.83,20,4,2026-04-16T07:25:14.671Z
6,13637925.00,3350,611,22320.66,20,5,2026-04-16T07:25:14.671Z
9,9171168.00,3169,549,16705.22,20,6,2026-04-16T07:25:14.671Z
8,7599844.00,3008,547,13893.68,20,7,2026-04-16T07:25:14.671Z
10,5536692.00,3398,602,9197.16,20,8,2026-04-16T07:25:14.671Z
2,3414168.00,3145,584,5846.18,20,9,2026-04-16T07:25:14.671Z
5,1064416.50,3367,600,1774.03,20,10,2026-04-16T07:25:14.671Z


In [0]:
# Write monthly revenue to gold
df_monthly.write \
    .format('delta') \
    .mode('overwrite') \
    .save(gold_revenue)

print('Gold monthly_revenue written!')

# Write top products to gold
df_products.write \
    .format('delta') \
    .mode('overwrite') \
    .save(gold_products)

print('Gold top_products written!')

Gold monthly_revenue written!
Gold top_products written!


In [0]:
# Read back and verify
df_rev_check  = spark.read.format('delta').load(gold_revenue)
df_prod_check = spark.read.format('delta').load(gold_products)

print(f'Monthly revenue rows: {df_rev_check.count()}')
print(f'Top products rows: {df_prod_check.count()}')

print('--- Monthly Revenue Sample ---')
display(df_rev_check.orderBy('order_year', 'order_month').limit(10))

print('--- Top Products ---')
display(df_prod_check.orderBy('rank'))

Monthly revenue rows: 96
Top products rows: 10
--- Monthly Revenue Sample ---


order_year,order_month,region,total_revenue,order_count,avg_order_value,unique_customers,total_units_sold,report_month,load_ts
2022,1,East,5975106.00,90,66390.07,20,512,2022-01,2026-04-16T07:26:25.962Z
2022,1,North,2292276.50,51,44946.60,17,293,2022-01,2026-04-16T07:26:25.962Z
2022,1,South,4175829.00,47,88847.43,18,268,2022-01,2026-04-16T07:26:25.962Z
2022,1,West,3239373.00,30,107979.10,15,187,2022-01,2026-04-16T07:26:25.962Z
2022,2,East,5063232.50,97,52198.27,20,571,2022-02,2026-04-16T07:26:25.962Z
2022,2,North,4319403.50,43,100451.24,19,249,2022-02,2026-04-16T07:26:25.962Z
2022,2,South,4886433.50,50,97728.67,17,284,2022-02,2026-04-16T07:26:25.962Z
2022,2,West,816040.00,24,34001.67,15,133,2022-02,2026-04-16T07:26:25.962Z
2022,3,East,6761159.00,96,70428.74,19,548,2022-03,2026-04-16T07:26:25.962Z
2022,3,North,3692764.00,42,87922.95,18,245,2022-03,2026-04-16T07:26:25.962Z


--- Top Products ---


product_id,total_revenue,total_units_sold,order_count,avg_order_value,unique_buyers,rank,load_ts
1,218964750.00,3223,562,389616.99,20,1,2026-04-16T07:26:33.947Z
4,65435040.00,3290,600,109058.40,20,2,2026-04-16T07:26:33.947Z
7,55411200.00,3398,605,91588.76,20,3,2026-04-16T07:26:33.947Z
3,22850125.00,2975,536,42630.83,20,4,2026-04-16T07:26:33.947Z
6,13637925.00,3350,611,22320.66,20,5,2026-04-16T07:26:33.947Z
9,9171168.00,3169,549,16705.22,20,6,2026-04-16T07:26:33.947Z
8,7599844.00,3008,547,13893.68,20,7,2026-04-16T07:26:33.947Z
10,5536692.00,3398,602,9197.16,20,8,2026-04-16T07:26:33.947Z
2,3414168.00,3145,584,5846.18,20,9,2026-04-16T07:26:33.947Z
5,1064416.50,3367,600,1774.03,20,10,2026-04-16T07:26:33.947Z
